← [Overview](../00_overview.ipynb)

# Partitional clustering: k-means and k-medoids

Partitional clustering assigns every period to **exactly one** cluster and describes each
cluster by a single representative period. The two methods below share the same two
ingredients — the **distance** of section 2 and the rule *"assign every period to its nearest
representative"* — and differ only in **how they choose the representatives**:

| Method | Representatives chosen to… | How it is solved |
|---|---|---|
| **k-means** | minimise the total within-cluster distance $J$ (*approximately*) | Lloyd's heuristic → local optimum |
| **k-medoids** | minimise that *same* $J$, centres restricted to real periods (*exactly*) | MILP → global optimum (needs a solver) |

So k-means and k-medoids chase the **same objective** — one only approximately, the other to
proven optimality. This notebook is about **how each formulates and solves that problem**.


---

## 1  The data: six periods as points

The [preprocessing notebook](../01_preprocessing.ipynb) already normalized each attribute to $[0, 1]$ and **unstacked** the flat
series so each of the six days becomes one row-vector.

Each row is one period: a single point in the $N_a \cdot N_t = 2 \times 4 = 8$-dimensional
feature space. Clustering groups these six points.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

import tsam
from tsam import ClusterConfig

pio.renderers.default = "notebook_connected"

# Preprocessed period matrix D (normalised + unstacked) from 01_preprocessing.
tiny_period_df = pd.read_csv(
    "../../../data/tiny_periods.csv", header=[0, 1], index_col=0
)
tiny_period_array = tiny_period_df.values  # shape (6, 8): six periods, eight features
N_PERIODS = tiny_period_array.shape[0]
N_ATTRS, N_TIMESTEPS = 2, 4

print("period matrix D:", tiny_period_array.shape)
tiny_period_df.round(4)

---

## 2  The objective: the distance that gets minimised in k-means and k-medoids

The shared building block of k-means and k-medoids is the **distance between a period and a
centre**. Writing $x_{p,a,t}$ for the value of attribute $a$ at timestep $t$ in period $p$:

$$
\text{dist}(x_p, c_k) = \sqrt{\sum_{a=1}^{N_a} \sum_{t=1}^{N_t} (x_{p,a,t} - c_{k,a,t})^2}
$$

The double sum walks **every coordinate** of the period vector:

* $a = 1 \dots N_a$ indexes the **attributes** — here $N_a = 2$ (`solar`, `load`);
* $t = 1 \dots N_t$ indexes the **timesteps within a period** — here $N_t = 4$ (the four
  6-hourly steps of a day).

so it runs over all $N_a \cdot N_t = 2 \times 4 = 8$ coordinates — exactly the eight columns
of $D$.

This is one **element-wise** subtraction of the two 8-vectors: coordinate $(a,t)$ of the
period is only ever compared with the **same** $(a,t)$ of the centre — `solar@t1` against
`solar@t1` So $x_p$ and $c_k$ are points of the same shape, and a centre is directly comparable to
a period whether it is a synthetic mean or a real day.

From this single distance, the quality of a whole clustering is the **total within-cluster
distance** $J$ — every period summed against the centre of the cluster it lands in:

$$
J = \sum_{k=1}^{N_k} \sum_{p \in \mathbb{C}_k} \text{dist}(x_p, c_k)^2
$$

| Symbol | Meaning |
|---|---|
| $x_p$ | period $p$ — one row of $D$ (`D_arr[p]`), a point in 8-D space |
| $\mathbb{C}_k$ | **cluster $k$**: the set of periods assigned to group $k$ |
| $c_k$ | the **centre** of cluster $k$ |
| $N_k$ | number of clusters ( = `n_clusters`) |

$J$ is the yardstick k-means and k-medoids both drive down.

In [ ]:
def euclidean_dist(x_p, c_k):
    """Euclidean distance between a period and a cluster center.

    x_p, c_k : np.ndarray, shape (N_a * N_t,) == (8,)
        A period and a center — same shape and coordinate order, one entry per
        (attribute a, timestep t) pair.

    Returns the scalar sqrt(sum((x_p - c_k) ** 2)); zero only for identical vectors.
    """
    return float(np.sqrt(np.sum((x_p - c_k) ** 2)))


# A concrete pair: period 0 vs a stand-in centre (period 2).
x_p = tiny_period_array[0]  # day 0 (a sunny day)
c_k = tiny_period_array[2]  # day 2 (an overcast day), used here as a stand-in centre

# Label every coordinate with the (a, t) it belongs to — the 8 columns of D.
terms = pd.DataFrame(
    {
        "a (attribute)": tiny_period_df.columns.get_level_values(0),
        "t (timestep)": tiny_period_df.columns.get_level_values(1).astype(int),
        "x_p": x_p,
        "c_k": c_k,
        "gap = x_p - c_k": x_p - c_k,
        "gap**2": (x_p - c_k) ** 2,
    }
)
print(terms.round(3).to_string(index=False))

# One term of the double sum picked out, e.g. a = solar, t = 1 (coordinate index 1):
print(
    "\nExample single term  a=solar, t=1:",
    f"(x = {x_p[1]:.3f} - c = {c_k[1]:.3f})**2 = {(x_p[1] - c_k[1]) ** 2:.3f}",
)
print(
    "sum over all",
    N_ATTRS * N_TIMESTEPS,
    "terms      =",
    round(float(((x_p - c_k) ** 2).sum()), 4),
)
print("dist(x_p, c_k) = sqrt(sum)        =", round(euclidean_dist(x_p, c_k), 4))

---

## 3  Approximate solution — k-means (Lloyd's algorithm)

**Mechanism:** Lloyd's algorithm does not solve the objective exactly; it converges to a
*local* optimum by alternating two cheap steps until assignments stop changing:

1. Initialise $k$ centres $c_1, \dots, c_k$ (e.g. k-means++).
2. **Assignment step:** put each period with its nearest centre,
   $\text{cluster}(p) = \arg\min_k \text{dist}(x_p, c_k)$.
3. **Update step:** move each centre to the mean of its members,
   $c_k = \frac{1}{|\mathbb{C}_k|} \sum_{p \in \mathbb{C}_k} x_p$.
4. Repeat from step 2.

Its default representative is therefore the cluster **mean** — a synthetic centroid that need
not match any real day.

**TSAM configuration for k-means:**

In [ ]:
# K-means: feature-based clustering using Lloyd's algorithm.
# representation defaults to 'mean' (centroid) for kmeans.
cfg_kmeans = ClusterConfig(method="kmeans", representation="mean")
print(cfg_kmeans)

### From-scratch Lloyd iteration on the tiny series

With the distance function in hand, the assignment step is just "call `euclidean_dist` for
every period against every centre and take the nearest", and the update step is the
centroid mean. Tracing it on the six periods (deterministic start at days 0, 2, 4, $k=3$):

In [ ]:
# Reuse euclidean_dist(x_p, c_k) defined above.
n_clusters = 3
# Deterministic initialisation: pick periods 0, 2, 4 as initial centers
centers = tiny_period_array[[0, 2, 4]].copy().astype(float)

print("Initial centers (periods 0, 2, 4):")
for k, center in enumerate(centers):
    print(f"  c{k} = {center.round(3)}")

for iteration in range(6):
    # --- Step 1: assign each period to its nearest center ---
    distances = np.empty((N_PERIODS, n_clusters))
    for period_idx in range(N_PERIODS):
        for k in range(n_clusters):
            distances[period_idx, k] = euclidean_dist(
                tiny_period_array[period_idx], centers[k]
            )

    assignments = np.empty(N_PERIODS, dtype=int)
    for period_idx in range(N_PERIODS):
        assignments[period_idx] = np.argmin(distances[period_idx])

    # --- Step 2: recompute each center as the mean of its assigned periods ---
    new_centers = np.empty_like(centers)
    for k in range(n_clusters):
        cluster_members = []
        for p in range(N_PERIODS):
            if assignments[p] == k:
                cluster_members.append(tiny_period_array[p])

        if len(cluster_members) > 0:
            new_centers[k] = np.mean(cluster_members, axis=0)
        else:
            new_centers[k] = centers[k]  # keep old center if no period was assigned

    converged = np.allclose(centers, new_centers)
    centers = new_centers
    print(f"\nIteration {iteration + 1}: assignments = {assignments}")
    if converged:
        print("  Converged.")
        break

print("\nFinal cluster assignments:")
for period_idx in range(N_PERIODS):
    print(f"  day_{period_idx} -> cluster {assignments[period_idx]}")

In [ ]:
# Verify centroid formula: c_k = (1/|C_k|) * sum of members
cluster_0_members = tiny_period_array[assignments == 0]
centroid_0 = cluster_0_members.mean(axis=0)

print("Cluster 0 members:")
for p in np.where(assignments == 0)[0]:
    d = euclidean_dist(tiny_period_array[p], centroid_0)
    print(f"  day_{p}: dist to centroid = {d:.4f}")

manual_centroid = cluster_0_members.sum(axis=0) / len(cluster_0_members)
print(f"\nCentroid c_0 (mean of members): {centroid_0.round(4)}")
print(f"Manual sum/count check:         {manual_centroid.round(4)}")
print("Match:", np.allclose(centroid_0, manual_centroid))

In [ ]:
# Raw tiny series — for the tsam.aggregate calls (it normalises internally).
tiny = pd.read_csv("../../../data/tiny.csv", index_col=0, parse_dates=True)
# tsam k-means on the tiny series (k=3) — reproduces the hand-traced partition.
result_km = tsam.aggregate(
    tiny, n_clusters=3, period_duration="1D", cluster=ClusterConfig(method="kmeans")
)
print(
    "k-means assignments:",
    np.asarray(result_km.cluster_assignments),
    "  weighted RMSE:",
    round(result_km.accuracy.weighted_rmse, 4),
)

**Reading the assignment array.** `cluster_assignments` has one entry per period, in period
order: entry `p` is the cluster that **day `p`** landed in. So `[1 1 2 2 0 0]` reads
"days 0–1 → cluster 1, days 2–3 → cluster 2, days 4–5 → cluster 0" — exactly the three
shape-pairs from the legend (sunny / overcast / cloudy). The cluster **labels** (0/1/2) are
arbitrary and can differ between methods or runs; what matters is the **partition** — *which*
days share a cluster. That is why the from-scratch trace above (labels 0/1/2) and tsam here
(labels 1/2/0) describe the *same* grouping.



## 4  Exact solution — k-medoids (MILP)

**Mechanism:** where Lloyd's algorithm only *approximates* the minimum of $J$, k-medoids
**solves it exactly**. Restricting every centre to be an **actual period** turns the search
into a finite combinatorial problem — the classic **$p$-median / facility-location** problem
(known in spatial planning as the *Hess model*) — which can be written as a **Mixed-Integer
Linear Program (MILP)** and handed to a solver (here HiGHS) for a *provably globally optimal*
clustering.

### The model

The only data the solver needs is the matrix of **pairwise distances** $d_{i,j}$ between
periods. A single family of binary variables encodes the whole clustering:

$$
z_{i,j} = \begin{cases} 1 & \text{period } j \text{ is assigned to centre } i \\ 0 & \text{otherwise} \end{cases}
$$

The diagonal $z_{i,i}=1$ marks period $i$ as one of the **open** centres (a medoid). The
program minimises the total within-cluster distance,

$$
\min_{z}\ \sum_{i}\sum_{j} d_{i,j}\, z_{i,j} \;=\; J,
$$

subject to three constraints — each one line of `_setup_k_medoids` in `k_medoids_exact.py`:

| # | constraint | meaning | rule in `_setup_k_medoids` |
|---|---|---|---|
| 1 | $\sum_i z_{i,j} = 1\ \ \forall j$ | every period is assigned to exactly one centre | `candToClusterRule` |
| 2 | $\sum_i z_{i,i} = k$ | exactly $k$ centres are opened | `noClustersRule` |
| 3 | $z_{i,j} \le z_{i,i}\ \ \forall i,j$ | a period may only be assigned to an **open** centre | `clusterRelationRule` |

### What $i$, $j$, $z$, $d$ mean on our six days

Everything above is concrete on the tiny dataset:

* $i$ and $j$ **both run over the six days** $0, 1, \dots, 5$ — every day is at once a *candidate
  centre* (index $i$) and a *period to be assigned* (index $j$).
* $d_{i,j}$ is simply the **distance between day $i$ and day $j$** — one cell of the $6\times6$
  matrix below. For example $d_{0,2}\approx 0.89$ is exactly the `dist(day 0, day 2) = 0.8851`
  worked out in section 2.
* $z_{i,j}=1$ reads "**day $j$ is represented by day $i$**"; the diagonal $z_{i,i}=1$ flags day
  $i$ as one of the three chosen **medoids**.

So the optimiser never looks at solar/load profiles — it works **from the distance matrix
alone**, choosing which 3 days act as centres and attaching every day to the cheapest one. A
caveat lives in the formulation itself: there are $n^2$ binary variables for $n$ periods, so
the exact route is practical for hundreds of periods, not tens of thousands.

**TSAM configuration for k-medoids:**

In [ ]:
# K-medoids: each representative is an actual observed period (medoid).
# Uses MILP optimization — solver='highs' (default, open-source).
cfg_kmedoids = ClusterConfig(method="kmedoids", representation="medoid", solver="highs")
print(cfg_kmedoids)

### The solver's only input: the distance matrix

The exact-MILP k-medoids never sees the day profiles at all — its solver receives **only**
the matrix of pairwise distances $d_{i,j}$ (`M.d` inside `_setup_k_medoids`). Below we build
that full $6\times6$ matrix for the tiny dataset and hand it straight to tsam's own model
builder and solver, then read the optimal medoids and assignment back out.

In [ ]:
# Build the 6x6 pairwise distance matrix — exactly the input `M.d` the MILP receives.
Dmat = np.array(
    [
        [
            euclidean_dist(tiny_period_array[i], tiny_period_array[j])
            for j in range(N_PERIODS)
        ]
        for i in range(N_PERIODS)
    ]
)

# Hand it to tsam's *own* k-medoids model builder + solver. No day profiles involved.
from tsam.algorithms.k_medoids_exact import _setup_k_medoids, _solve_given_pyomo_model

model = _setup_k_medoids(Dmat, n_clusters=3)
r_x, r_y, r_obj = _solve_given_pyomo_model(model, solver="highs")

medoids = [
    p for p, opened in enumerate(r_y) if opened == 1
]  # the periods with z_ii = 1
assignment = r_x.argmax(axis=1)  # period j -> its medoid i

print("opened medoids (z_ii = 1):      ", medoids)
print("each period -> its medoid:      ", assignment.tolist())
print("objective  sum d_ij * z_ij   =  ", round(r_obj, 4))

In [ ]:
labels = [f"day_{p}" for p in range(N_PERIODS)]
fig = px.imshow(
    Dmat,
    x=labels,
    y=labels,
    text_auto=".2f",
    color_continuous_scale="Blues",
    labels={"x": "candidate centre", "y": "period", "color": "distance"},
    title="Pairwise distance matrix — the exact input to the k-medoids optimiser",
)

# Outline the cell each period contributes: period j -> the medoid the MILP assigned it to.
for j in range(N_PERIODS):
    m = int(assignment[j])
    fig.add_shape(
        type="rect",
        x0=m - 0.5,
        x1=m + 0.5,
        y0=j - 0.5,
        y1=j + 0.5,
        line={"color": "#EF553B", "width": 3},
    )
fig.update_layout(width=560, height=520)
fig.show()

obj = sum(Dmat[j, int(assignment[j])] for j in range(N_PERIODS))
print(f"Objective = sum of outlined cells (day -> its medoid) = {obj:.3f}")

Both axes are indexed by the same `day_0 … day_5` from the legend, so a cell `(i, j)` is the
distance between two specific days. The block structure is the whole story: days 0–1 (sunny),
2–3 (overcast) and 4–5 (cloudy) sit cheaply close (dark), while crossing between blocks is
expensive (light). The optimiser **opens one centre per block** ($z_{i,i}=1$, the medoids) and
**assigns every period to it** ($z_{i,j}=1$, the outlined cells) — the explicit form of the
abstract "$p \in \mathbb{C}_k$" membership from section 2. Here the opened medoids `[0, 3, 5]`
are exactly one day from each shape-block: a sunny day (0), an overcast day (3) and the
cloudy/extreme-load day (5). It minimises $\sum_{i,j} d_{i,j}\, z_{i,j}$, the sum of the
outlined cells: here $\approx 1.02$ (the diagonal *day → itself* terms are zero).

Notice the optimum can be **degenerate**: inside a symmetric pair either day is an equally
good medoid, so the solver's particular pick is an arbitrary tie-break — but the **partition**
into pairs is unique and globally optimal.

In [ ]:
# tsam k-medoids on the tiny series (k=3).
# Raw tiny series — for the tsam.aggregate calls (it normalises internally).
tiny = pd.read_csv("../../../data/tiny.csv", index_col=0, parse_dates=True)
print("Raw tiny series shape:", tiny)
result_kmed = tsam.aggregate(
    tiny, n_clusters=3, period_duration="1D", cluster=ClusterConfig(method="kmedoids")
)
print(
    "k-medoids assignments:",
    np.asarray(result_kmed.cluster_assignments),
    "  weighted RMSE:",
    round(result_kmed.accuracy.weighted_rmse, 4),
)

---

**Up next:**
* [Extremal-prototype selection](03_extremal_prototype_selection.ipynb) — k-maxoids, the third partitional method, which maximises spread instead of minimising $J$
* [Agglomerative clustering](02_agglomerative_clustering.ipynb) — hierarchical Ward and contiguous Ward
* [Comparing clustering methods](../../../tutorials/comparing_clustering_methods.ipynb) — watch the methods reach *different* partitions on the same data
* [Representation](../03_representation.ipynb) — choosing what the cluster center looks like
* [Extreme periods](../04_extreme_periods.ipynb) — preserving peaks through clustering